# Editor de Vídeo — Colab

O editor é puxado automaticamente de `ProfAllanIFBA/Editor_de_Video`. No Colab, `preparar_preview="auto"` cria automaticamente uma cópia leve/faststart só para o player; o render final continua usando o vídeo original.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.request

from google.colab import files

REPO = "ProfAllanIFBA/Editor_de_Video"
BRANCH = "main"
ARQUIVO_EDITOR = "marcador_cortes_jupyter.py"
PASTA = Path("/content/Editor_de_Video")
PASTA.mkdir(parents=True, exist_ok=True)
os.chdir(PASTA)

print("=== Preparando ambiente ===")

# FFmpeg / ffprobe
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    print("Instalando FFmpeg...")
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])
else:
    print("FFmpeg: OK")

# Bibliotecas Python
pacotes = []
if importlib.util.find_spec("faster_whisper") is None:
    pacotes.append("faster-whisper")
if importlib.util.find_spec("PIL") is None:
    pacotes.append("pillow")
if pacotes:
    print("Instalando:", ", ".join(pacotes))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--timeout", "1000", "--retries", "10", *pacotes
    ])
else:
    print("Dependências Python: OK")

# Puxa sempre o editor publicado no GitHub.
url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/{ARQUIVO_EDITOR}"
destino = PASTA / ARQUIVO_EDITOR
try:
    req = urllib.request.Request(
        url,
        headers={"Cache-Control": "no-cache", "Pragma": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as resposta:
        destino.write_bytes(resposta.read())
    print("Editor atualizado do GitHub:", url)
except Exception as erro:
    print("Não foi possível baixar do GitHub:", erro)
    print("Selecione agora marcador_cortes_jupyter.py no seu computador.")
    enviados = files.upload()
    candidatos = [nome for nome in enviados if nome.lower().endswith(".py")]
    if not candidatos:
        raise RuntimeError("Nenhum arquivo .py foi enviado.")
    origem = Path(candidatos[0])
    if origem.resolve() != destino.resolve():
        shutil.copy2(origem, destino)

# Importa diretamente pelo caminho, sem reaproveitar versão antiga da sessão.
importlib.invalidate_caches()
sys.modules.pop("marcador_cortes_jupyter", None)
spec = importlib.util.spec_from_file_location("marcador_cortes_jupyter", destino)
mc = importlib.util.module_from_spec(spec)
sys.modules["marcador_cortes_jupyter"] = mc
spec.loader.exec_module(mc)

print("\nEditor pronto.")
print("Versão:", mc.__version__)
print("Arquivo:", mc.__file__)

# 2. Vídeo principal

In [ ]:
print("Selecione o vídeo principal no seu computador.")
enviados = files.upload()

extensoes_video = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
candidatos = [PASTA / nome for nome in enviados if Path(nome).suffix.lower() in extensoes_video]
if not candidatos:
    raise RuntimeError("Nenhum arquivo de vídeo reconhecido foi enviado.")

VIDEO_ENTRADA = str(candidatos[0].resolve())
print("Vídeo:", VIDEO_ENTRADA)

# 3. Editor

In [ ]:
# Usa GPU para a transcrição quando o runtime do Colab disponibilizar NVIDIA/CUDA.
tem_cuda = False
if shutil.which("nvidia-smi"):
    teste = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    tem_cuda = teste.returncode == 0

dispositivo = "cuda" if tem_cuda else "cpu"
compute_type = "float16" if tem_cuda else "int8"

video_path = Path(VIDEO_ENTRADA)
transcricao_json = video_path.with_name(f"{video_path.stem}_transcricao.json")
modo_transcricao = "auto" if transcricao_json.exists() else True

print("Transcrição:", dispositivo, "/", compute_type)
if not tem_cuda:
    print("GPU NVIDIA não detectada neste runtime; a transcrição usará CPU.")

mc.marcador_cortes(
    arquivo=VIDEO_ENTRADA,
    largura=1600,
    preparar_preview="auto",   # no Colab: preview leve + faststart automaticamente
    transcricao=modo_transcricao,
    modelo_transcricao="small",
    idioma_transcricao="pt",
    dispositivo_transcricao=dispositivo,
    compute_type_transcricao=compute_type,
    pausa_bloco=0.7,
    projeto="auto",
    legendas=False,
)

# 4. Renderizar

O preview leve existe apenas para a interface do Colab. `processar_projeto()` continua renderizando a partir de `VIDEO_ENTRADA`, ou seja, do vídeo original.

In [ ]:
from IPython.display import Video, display

video_path = Path(VIDEO_ENTRADA)
projeto = video_path.with_name(f"{video_path.stem}_projeto_editor.json")
saida = video_path.with_name(f"{video_path.stem}_editado.mp4")

saida = mc.processar_projeto(
    entrada=VIDEO_ENTRADA,
    projeto=projeto,
    saida=saida,
)

print("Criado em:", saida)
display(Video(str(saida), embed=False, width=700))

In [ ]:
# Execute quando quiser baixar o MP4 final para o computador.
files.download(str(saida))